In [1]:
# ── 0. Install libraries ──────────────────────────────────────────
# In Kaggle/Colab, run this once if the packages are missing.
# For final offline runs, the packages and model weights must already be available/cached.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate pillow pandas tqdm
!pip install -q -U "bitsandbytes>=0.46.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.4 MB/s eta 0:00:00


In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Auto-detects folder containing train.csv, val.csv, and test.csv.
#def find_data_dir():
#    candidates = [
#        Path("/kaggle/input/competitions/pixels-to-predictions")
#    ]
#    for base in candidates:
#        if (base / "train.csv").exists() and (base / "val.csv").exists() and (base / "test.csv").exists():
#           return base
#        if base.exists():
#            for child in base.iterdir():
#                if child.is_dir() and (child / "train.csv").exists() and (child / "val.csv").exists() and (child / "test.csv").exists():
#                    return child
#   raise FileNotFoundError("Could not find train.csv/val.csv/test.csv.")

DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions")
#IMAGE_ROOT = DATA_DIR / "images"
print("DATA_DIR =", DATA_DIR)

ADAPTER_DIR = "/kaggle/input/datasets/tanvirshahriar0/adapter-save"

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
MODEL_ID = "NA"

# Used local model dir for this assignment.
# Online model untested for this notebook.
LOCAL_MODEL_DIR = Path("/kaggle/input/datasets/tanvirshahriar0/smolvlm-500m-instruct/SmolVLM-500M-Instruct")
MODEL_PATH = str(LOCAL_MODEL_DIR) if LOCAL_MODEL_DIR.exists() else MODEL_ID
print("MODEL_PATH =", MODEL_PATH)

# ── Main knobs ────────────────────────────────────────────────────────────────
IMG_SIZE = 224 # 248 for full run worked.
MAX_LENGTH = 768               # text truncation length
TRAIN_BATCH_SIZE = 1           # safe on free GPUs
EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1           # effective batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS xxx adjust if loss unstable
EPOCHS = 1                     # start with 1; try 2 if time allows
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
MAX_TRAIN_STEPS = 1500         # cap for quicker training? # saw quick drop in loss, so cap to prevent overfitting

MAX_LECTURE_CHARS = 300
MAX_HINT_CHARS = 200
#MAX_QUESTION_CHARS = 500

# LoRA/DoRA settings.
LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
USE_DORA = True
MAX_TRAINABLE_PARAMS = 5000000

# Train-time PIL augmentation. Mild.
USE_IMAGE_AUGMENTATION = True #False for full run.

# ── Device info ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


2026-05-07 05:38:25.937228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778132306.157747      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778132306.219385      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778132306.731039      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778132306.731081      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778132306.731084      23 computation_placer.cc:177] computation placer alr

DATA_DIR = /kaggle/input/competitions/pixels-to-predictions
MODEL_PATH = /kaggle/input/datasets/tanvirshahriar0/smolvlm-500m-instruct/SmolVLM-500M-Instruct
Using device: cuda
GPU: Tesla T4


## 2. Load and preprocess data

In [3]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# 'choices' column.
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
display(train_df.head(2))

# Sanity checks
assert "answer" in train_df.columns and "answer" in val_df.columns
assert "answer" not in test_df.columns or test_df["answer"].isna().all()
assert train_df["num_choices"].min() >= 2
assert train_df["num_choices"].max() <= 10


Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

# Columns that are useful context but not too noisy.
META_COLS = ["subject", "topic", "category", "skill", "grade"]

#def clean_text(x):
#    if pd.isna(x):
#        return ""
#    return str(x).strip()

# end truncation of lecture.
"""
def clean_text(x, max_chars=None):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if max_chars is None or len(x) <= max_chars:
        return x

    # Keep beginning and end instead of only the beginning
    half = max_chars // 2
    return x[:half] + " ... " + x[-half:]
"""

# Middle truncation.
def clean_text(x, max_chars=None):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if max_chars is None or len(x) <= max_chars:
        return x

    keep_each_side = max_chars // 2
    return x[:keep_each_side] + " ... " + x[-keep_each_side:]

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for SmolVLM.
    The <image> token tells the model where the image should be used.
    """
    meta_parts = []
    for col in META_COLS:
        if col in row and clean_text(row[col]):
            meta_parts.append(f"{col}: {clean_text(row[col])}")

    context_parts = []
    
    lecture = clean_text(row.get("lecture", ""), MAX_LECTURE_CHARS)
    hint = clean_text(row.get("hint", ""), MAX_HINT_CHARS)
    question = clean_text(row.get("question", ""))  # no max, keeps full question
    
    if lecture:
        context_parts.append(f"Lecture: {lecture}")
    if hint:
        context_parts.append(f"Hint: {hint}")
    
    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    
    prompt = "<image>\n"
    prompt += "You are answering a science multiple-choice question. Use the image and text.\n"
    
    if meta_parts:
        prompt += "Metadata: " + " | ".join(meta_parts) + "\n"
    
    if context_parts:
        prompt += "\n".join(context_parts) + "\n"
    
    prompt += f"Question: {question}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "The correct answer is:"

    #if include_answer:
    #    answer_idx = int(row["answer"])
    #    prompt += f" {CHOICE_LETTERS[answer_idx]}"
    #return prompt

    # Train on letter + choice text instead of just letter.
    if include_answer:
        answer_idx = int(row["answer"])
        answer_letter = CHOICE_LETTERS[answer_idx]
        answer_text = choices[answer_idx]
        prompt += f" {answer_letter}. {answer_text}"
    return prompt


print(build_prompt(train_df.iloc[0], include_answer=True))


<image>
You are answering a science multiple-choice question. Use the image and text.
Metadata: subject: natural science | topic: literacy-in-science | category: Adaptations and natural selection | skill: How can animal behaviors affect reproductive success? Identify evidence to support a claim | grade: grade8
Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring ... attract or compete for mates won't always successfully mate and reproduce, and offspring that are fed and protected won't always survive to adulthood.
Hint: Animals often behave in certain ways that can increase their reproductive success. Read the passage  ... o develop into a frog.
Figure: an Amazonian poison frog carrying a dark-colored tadpole on his back.
Question: Why might putting each tadpole in its own pool of water increase the reproductive success of a male Amazonian poison frog? Complete the claim below tha

In [5]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
def get_image_path(data_dir: Path, rel_path: str) -> Path:
    image_path = str(rel_path)

    # CSV example: images/train/train_07667.png
    candidate = data_dir / "images" / image_path
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f"Could not find image: {candidate}")

def augment_image(img: Image.Image) -> Image.Image:
    """Small augmentations."""
    if random.random() < 0.5:
        angle = random.uniform(-3, 3)
        img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=(255, 255, 255))
    if random.random() < 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.1))
    return img

class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 384, is_train: bool = True, augment: bool = False):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img_path = get_image_path(self.data_dir, rel_path)
        img = Image.open(img_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        if self.augment:
            img = augment_image(img)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image = self._load_image(row["image_path"])
        prompt = build_prompt(row, include_answer=False)

        item = {
            "id": row["id"],
            "image": image,
            "prompt": prompt,
            "choices": row["choices"],
            "num_choices": int(row["num_choices"]),
        }
        if "answer" in row and pd.notna(row["answer"]):
            item["answer"] = int(row["answer"])
            item["text"] = build_prompt(row, include_answer=True)
        return item

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True, augment=USE_IMAGE_AUGMENTATION)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False, augment=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False, augment=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
print("One image size:", train_ds[0]["image"].size)


Datasets created: train=3109, val=1048, test=1008
One image size: (248, 248)


In [ ]:
"""#####################F #This is for loading adapter weights.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(LOCAL_MODEL_DIR)

base_model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

model.eval()"""

'#####################F #FFFFFF\nimport torch\nfrom transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig\nfrom peft import PeftModel\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type="nf4",\n    bnb_4bit_compute_dtype=torch.float16,\n    bnb_4bit_use_double_quant=True,\n)\n\nprocessor = AutoProcessor.from_pretrained(LOCAL_MODEL_DIR)\n\nbase_model = AutoModelForImageTextToText.from_pretrained(\n    LOCAL_MODEL_DIR,\n    quantization_config=bnb_config,\n    device_map="auto",\n    dtype=torch.float16,\n)\n\nmodel = PeftModel.from_pretrained(\n    base_model,\n    ADAPTER_DIR,\n    is_trainable=False,\n)\n\nmodel.eval()'

## 3. Load SmolVLM with QLoRA/DoRA

The base model is loaded in 4-bit NF4 to save memory. Only adapter weights are trainable.


In [7]:
# ── 3a. Load processor and base model ────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_PATH)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    dtype=compute_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

if not torch.cuda.is_available():
    model.to(device)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

model.config.use_cache = False

print("Model loaded.")

Model loaded.


In [8]:
# ── 3b. Attach LoRA/DoRA adapter ──────────────────────────────────────────────
if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
# These names are used in many SmolVLM/SmolLM blocks. PEFT applies LoRA to matching modules.
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",        # attention
    "gate_proj", "up_proj", "down_proj",          # MLP
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none", # max parameter limit consideration.
    target_modules=target_modules,
    use_dora=USE_DORA,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = count_trainable_params(model)
print(f"Trainable parameters: {trainable_params:,}")
assert trainable_params <= MAX_TRAINABLE_PARAMS, (
    # We can Lower LORA_R or reduce target_modules.
    f"Too many trainable parameters: {trainable_params:,} > {MAX_TRAINABLE_PARAMS:,}. "
)


trainable params: 2,696,192 || all params: 510,178,496 || trainable%: 0.5285
Trainable parameters: 2,696,192


## 4. Fine-tuning

In [9]:
# ── 4a. Collator ─────────────────────────────────────────────────────────────
def train_collate_fn(batch):
    texts = [x["text"] for x in batch]        # prompt + answer
    prompts = [x["prompt"] for x in batch]    # prompt only
    images = [x["image"] for x in batch]

    encoded = processor(
        text=texts,
        images=images,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )

    prefix_encoded = processor(
        text=prompts,
        images=images,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )

    labels = encoded["input_ids"].clone()

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100

    prefix_lengths = prefix_encoded["attention_mask"].sum(dim=1).tolist()
    for i, prefix_len in enumerate(prefix_lengths):
        labels[i, :int(prefix_len)] = -100

    encoded["labels"] = labels
    return encoded

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=train_collate_fn,
)

In [10]:
# ── 4b. Training loop ────────────────────────────────────────────────────────
model.train()

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
max_steps = EPOCHS * num_update_steps_per_epoch
if MAX_TRAIN_STEPS is not None:
    max_steps = min(max_steps, MAX_TRAIN_STEPS)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_warmup_steps = int(WARMUP_RATIO * max_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_steps,
)

print(f"Training for up to {max_steps} optimizer steps")

completed_steps = 0
running_loss = 0.0
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    pbar_total = (
        MAX_TRAIN_STEPS * GRAD_ACCUM_STEPS
        if MAX_TRAIN_STEPS is not None
        else len(train_loader)
    )
    
    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        total=min(len(train_loader), pbar_total),
    )
    
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            completed_steps += 1

            avg_loss = running_loss / max(1, completed_steps * GRAD_ACCUM_STEPS)
            pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

            if completed_steps >= max_steps:
                break

    if completed_steps >= max_steps:
        break

print("Training complete. Optimizer steps:", completed_steps)


Training for up to 3109 optimizer steps


Epoch 1/1:   0%|          | 0/3109 [00:00<?, ?it/s]

Training complete. Optimizer steps: 3109


## 5. Multiple-choice log-likelihood scoring

In [11]:
# ── 5a. Scoring helpers ──────────────────────────────────────────────────────

def move_to_model_device(batch):
    return {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }


def sequence_nll(logits, labels):
    """
    labels uses -100 for tokens we want to ignore.
    """
    # Causal LM rule:
    logits = logits[:, :-1, :]
    labels = labels[:, 1:]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        reduction="none",
        ignore_index=-100,
    )

    losses = losses.reshape(labels.shape)
    return losses.sum(dim=1)


@torch.inference_mode()
def score_choice_batch(prompts, images, answer_texts):
    """
    Score each candidate answer.

    We feed the model prompt + answer_letter

    We mask the prompt tokens so only the answer continuation is scored.
    """
    full_texts = [
        prompt + answer
        for prompt, answer in zip(prompts, answer_texts)
    ]

    # Full input: prompt + candidate answer
    full_inputs = processor(
        text=full_texts,
        images=images,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )

    # Prefix input: prompt only
    prefix_inputs = processor(
        text=prompts,
        images=images,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )

    prefix_lengths = prefix_inputs["attention_mask"].sum(dim=1).tolist()

    labels = full_inputs["input_ids"].clone()

    # Ignore padding tokens
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    # Ignore prompt tokens, score only answer tokens
    for i, prefix_len in enumerate(prefix_lengths):
        labels[i, :int(prefix_len)] = -100

    full_inputs = move_to_model_device(full_inputs)
    labels = labels.to(model.device)

    outputs = model(**full_inputs)
    scores = sequence_nll(outputs.logits, labels)

    return scores.cpu().numpy()


@torch.inference_mode()
def predict_dataset(dataset, batch_size=EVAL_BATCH_SIZE, show_progress=True):
    model.eval()

    all_ids = []
    all_preds = []
    targets = []

    starts = range(0, len(dataset), batch_size)
    if show_progress:
        starts = tqdm(starts, desc="Predicting")

    for start in starts:
        end = min(start + batch_size, len(dataset))
        items = [dataset[i] for i in range(start, end)]

        candidates = []
        num_choices_per_item = []
        
        for item in items:
            all_ids.append(item["id"])
            num_choices_per_item.append(item["num_choices"])
        
            if "answer" in item:
                targets.append(item["answer"])
        
            #for choice_idx in range(item["num_choices"]):
            #    candidates.append((
            #        item["prompt"],
            #        item["image"],
            #        " " + CHOICE_LETTERS[choice_idx],
            #    ))

            # scoring change for letter + choice text.
            for choice_idx in range(item["num_choices"]):
                answer_letter = CHOICE_LETTERS[choice_idx]
                answer_text = item["choices"][choice_idx]
            
                candidates.append((
                    item["prompt"],
                    item["image"],
                    f" {answer_letter}. {answer_text}",
                ))
        
        flat_prompts = [x[0] for x in candidates]
        flat_images  = [x[1] for x in candidates]
        flat_answers = [x[2] for x in candidates]
        
        flat_scores = score_choice_batch(flat_prompts, flat_images, flat_answers)
        
        offset = 0
        for num_choices in num_choices_per_item:
            scores_for_item = flat_scores[offset : offset + num_choices]
            all_preds.append(int(np.argmin(scores_for_item)))
            offset += num_choices

    result = pd.DataFrame({
        "id": all_ids,
        "answer": all_preds,
    })

    if len(targets) == len(all_preds):
        result["target"] = targets

    return result

In [ ]:
# ── 5b. Validate ─────────────────────────────────────────────────────────────
small_val_ds = torch.utils.data.Subset(val_ds, range(100))

val_pred_df = predict_dataset(small_val_ds, batch_size=1, show_progress=True)
val_acc = (val_pred_df["answer"] == val_pred_df["target"]).mean()

#val_pred_df = predict_dataset(val_ds, batch_size=EVAL_BATCH_SIZE)
#val_acc = (val_pred_df["answer"] == val_pred_df["target"]).mean()
print(f"Validation accuracy: {val_acc:.4f}")
display(val_pred_df.head())

Predicting:   0%|          | 0/100 [00:00<?, ?it/s]

Validation accuracy: 0.6100


,id,answer,target
0,val_00671,0,0
1,val_04111,2,1
2,val_02022,0,3
3,val_01237,0,0
4,val_03458,0,4


## 6. Save adapter and create Kaggle submission

In [13]:
# ── 6a. Save adapter ─────────────────────────────────────────────────────────
ADAPTER_DIR = Path("smolvlm_scienceqa_adapter")
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter and processor to {ADAPTER_DIR}")

Saved adapter and processor to smolvlm_scienceqa_adapter


In [14]:
# ── 6b. Predict test and write submission.csv ────────────────────────────────
test_pred_df = predict_dataset(test_ds, batch_size=EVAL_BATCH_SIZE, show_progress=True)
submission = test_pred_df[["id", "answer"]].copy()
submission["answer"] = submission["answer"].astype(int)

# Checks
submission = test_df[["id"]].merge(submission, on="id", how="left")
assert submission["answer"].notna().all(), "Some test ids did not receive predictions."
submission["answer"] = submission["answer"].astype(int)

submission.to_csv("submission.csv", index=False)
print(submission.shape)
display(submission.head())
print("Wrote submission.csv")

Predicting:   0%|          | 0/1008 [00:00<?, ?it/s]

(1008, 2)


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,0
4,test_00930,0


Wrote submission.csv


In [15]:
"""# TEST OF TEST.
small_test_ds = torch.utils.data.Subset(test_ds, range(min(5, len(test_ds))))

small_preds = predict_dataset(
    small_test_ds,
    batch_size=1,
    show_progress=True,
)

submission_small = small_preds[["id", "answer"]].copy()
submission_small["answer"] = submission_small["answer"].astype(int)

submission_small.to_csv("submission_small.csv", index=False)
submission_small.head()"""

'# TEST OF TEST.\nsmall_test_ds = torch.utils.data.Subset(test_ds, range(min(5, len(test_ds))))\n\nsmall_preds = predict_dataset(\n    small_test_ds,\n    batch_size=1,\n    show_progress=True,\n)\n\nsubmission_small = small_preds[["id", "answer"]].copy()\nsubmission_small["answer"] = submission_small["answer"].astype(int)\n\nsubmission_small.to_csv("submission_small.csv", index=False)\nsubmission_small.head()'

In [ ]:
# ── 6c. Zip adapter for submission ─────────────────────────────────────────

import shutil
from IPython.display import FileLink, display

adapter_dir = "/kaggle/working/smolvlm_scienceqa_adapter"
zip_path = "/kaggle/working/smolvlm_scienceqa_adapter.zip"

shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    adapter_dir
)

display(FileLink(zip_path))

/kaggle/working/smolvlm_scienceqa_adapter.zip